# Scrape Product Cards to CSV

Scrape product cards (name, price, rating, link) from a listing site across several pages and save them to CSV with pandas. The original scraped a business directory; this version uses the public practice site books.toscrape.com and scrapes politely.

## How to run
Run all cells in order. The data is saved as `books.csv`.

_Rewritten from my original Colab notebook: `web_scrap.ipynb`._

In [ ]:
!pip install -q requests beautifulsoup4 pandas

In [ ]:
BASE_URL = "https://books.toscrape.com/"
NUM_PAGES = 3

In [ ]:
import time
import urllib.robotparser
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "learning-scraper/1.0 (educational project)"}
DELAY_SECONDS = 1.5

robots = urllib.robotparser.RobotFileParser()
robots.set_url(urljoin(BASE_URL, "/robots.txt"))
robots.read()

def get_soup(url):
    """Download a page politely and return it parsed with BeautifulSoup."""
    if not robots.can_fetch(HEADERS["User-Agent"], url):
        print("robots.txt does not allow:", url)
        return None
    time.sleep(DELAY_SECONDS)  # be gentle with the server
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

## Extract data from one page

In [ ]:
def parse_page(soup, page_url):
    books = []
    for card in soup.select("article.product_pod"):
        link = card.select_one("h3 a")
        books.append({
            "title": link["title"],
            "price": card.select_one("p.price_color").get_text(strip=True),
            "rating": card.select_one("p.star-rating")["class"][1],  # e.g. 'Three'
            "url": urljoin(page_url, link["href"]),
        })
    return books

## Loop over pages and save

In [ ]:
import pandas as pd

all_books = []
for page in range(1, NUM_PAGES + 1):
    url = f"{BASE_URL}catalogue/page-{page}.html"
    print("Scraping", url)
    soup = get_soup(url)
    if soup:
        all_books.extend(parse_page(soup, url))

df = pd.DataFrame(all_books)
df.to_csv("books.csv", index=False)
print("Number of books:", len(df))
df.head()